# Grant Webscraper

## Project Structure

In [1]:
import os
import re
import hashlib
import datetime as _dt

import requests
from bs4 import BeautifulSoup
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from urllib.parse import urlparse, urlunparse

# ----------------------------------------------------
# 1) Paths and basic config (EDIT ONLY IF NEEDED)
# ----------------------------------------------------
EXCEL_PATH = r"C:\Users\miked\Desktop2\IConnectFoundation\grantMinded\logs\scored_results.xlsx"

CORPUS_HTML_DIR = r"C:\Users\miked\Desktop2\IConnectFoundation\grantMinded\logs\corpus\html"
CORPUS_TEXT_DIR = r"C:\Users\miked\Desktop2\IConnectFoundation\grantMinded\logs\corpus\text"

os.makedirs(CORPUS_HTML_DIR, exist_ok=True)
os.makedirs(CORPUS_TEXT_DIR, exist_ok=True)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/114.0.0.0 Safari/537.36"
    )
}

# ----------------------------------------------------
# 2) Your mission keyword weights + reference text
# ----------------------------------------------------
keyword_weights = {
    "aging": 3,
    "dementia": 3,
    "isolation": 4,
    "alzheimer's": 3,
    "telehealth": 2,
    "501(c)(3)": 2,
    "grant": 3,
    "funding": 3
}

_reference_text = (
    "health care programs and partnerships that improve outcomes and quality of life for older adults, "
    "people with disabilities, and caregivers. initiatives to align health and social care, community grants, 501 c 3. "
    "leveraging technology as a tool for connection, promoting meaningful engagement"
)

# ----------------------------------------------------
# 3) Helpers copied / adapted from your main code
# ----------------------------------------------------
def normalize_url(u):
    p = urlparse(u)
    p = p._replace(fragment="")
    return urlunparse((p.scheme.lower(), p.netloc.lower(), p.path, p.params, p.query, ""))

def doc_id_from_url(u):
    return hashlib.sha1(normalize_url(u).encode("utf-8")).hexdigest()[:16]

def html_to_clean_text(html):
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    text = soup.get_text(separator=" ", strip=True)
    return re.sub(r"\s+", " ", text).strip()

def content_sha1(text):
    return hashlib.sha1((text or "").encode("utf-8")).hexdigest()

def now_iso_utc():
    t = _dt.datetime.now(_dt.timezone.utc).replace(microsecond=0)
    return t.isoformat().replace("+00:00", "Z")

def keyword_score(text, keywords):
    score = 0
    matched = []
    t = (text or "").lower()
    for word, weight in keywords.items():
        pat = rf"\b{re.escape(word.lower())}\b"
        if re.search(pat, t):
            score += weight
            matched.append(word)
    return score, matched

def _simple_tokenizer(s):
    s = re.sub(r"[^a-z0-9\s]", " ", (s or "").lower())
    toks = s.split()
    return [t for t in toks if len(t) > 2]

def cosine_score(grant_text, ref_text=_reference_text):
    docs = [ref_text, grant_text or ""]
    vec = TfidfVectorizer(stop_words="english", tokenizer=_simple_tokenizer, token_pattern=None)
    m = vec.fit_transform(docs)
    sim = cosine_similarity(m[0:1], m[1:2])[0][0]
    return float(sim)

# ----------------------------------------------------
# 4) Your curated URLs
# ----------------------------------------------------
# curated_urls = [
#     "https://about.bankofamerica.com/en/making-an-impact/grant-funding-for-nonprofits-sponsorship-programs",
#     "https://grants.gov/search-results-detail/356414",
#     "https://alzfdn.org/find-a-member/grant-information/bi-annual-grant/",
#     "https://alzfdn.org/find-a-member/grant-information/brodsky-innovation-grant/",
#     "https://alzfdn.org/pligrant/",
#     "https://bobwoodrufffoundation.org/programs/network-partners/grants/bwf-grants/",
#     "https://curealz.org/the-research/grant-process/",
#     "https://eda.gov/funding/programs/american-rescue-plan/coal-communities-commitment",
#     "https://eda.gov/funding/programs/assistance-to-coal-communities",
#     "https://eda.gov/funding/programs/recompete-pilot-program",
#     "https://eda.gov/funding/programs/regional-technology-and-innovation-hubs",
#     "https://eda.gov/funding/programs/stem-challenge",
#     "https://grants.gov/search-results-detail/341131",
#     "https://grants.nih.gov/grants/guide/notice-files/NOT-AG-23-060.html",
#     "https://grants.nih.gov/grants/guide/pa-files/PAR-24-281.html",
#     "https://grantwatch.com/grant/170682/grants-to-usa-canada-and-international-research-institutions-for-studies-related-to-waste-management.html",
#     "https://grantwatch.com/grant/172348/award-to-an-ontario-teacher-of-grades-k-to-recognize-achievements-in-agricultural-education.html",
#     "https://grantwatch.com/grant/190382/grants-and-in-kind-services-to-usa-organizations-for-programs-that-promote-recycling.html",
#     "https://grantwatch.com/grant/197026/grants-to-alberta-organizations-and-businesses-to-strengthen-the-agriculture-industry.html",
#     "https://grantwatch.com/grant/197305/grants-to-alberta-producers-to-enhance-water-management.html",
#     "https://www.thegrantportal.com/grant-details/69958/grant-for-community-led-solutions-boosting-small-business-development?filter=1&search=69958",
#     "https://www.thegrantportal.com/grant-details/71226/grants-for-nonprofits-to-enhance-quality-of-peoples-lives?filter=1&search=71226",
#     "https://www.thegrantportal.com/grant-details/73123/grant- to-honor-the-communitys-best-nonprofit-organizations?filter=1&amp;search=73123",
#     "https://nnycf.org/grants/grant-information/",
#     "https://nnycf.org/micro-grant-program/",
#     "https://professional.heart.org/en/research-programs/aha-funding-opportunities/novel-ai-approaches-to-advance-cv-and-brain-health?utm_source=chatgpt.com",
#     "https://rifoundation.org/grant/community-partner-resilience-grants",
#     "https://www.afar.org/grants/small-research-grant-program-for-the-next-generation-of-researchers-in_ad",
#     "https://www.agingresearch.org/who-we-are/",
#     "https://www.alz.org/research/for_researchers/grants/about-our-grants",
#     "https://www.alz.org/research/for_researchers/grants/types-of-grants",
#     "https://www.alz.org/research/for_researchers/grants/types-of-grants/hsr-adrd",
#     "https://www.alzdiscovery.org/research-and-grants/funding-opportunities",
#     "https://www.christopherreeve.org/todays-care/get-support/grants-for-non-profits/program-overview/",
#     "https://www.communitywestfoundation.org/apply/",
#     "https://www.eda.gov/funding/disaster-recovery/supplemental/2019",
#     "https://www.eda.gov/funding/funding-opportunities/build-back-better-regional-challenge-fy21-american-rescue-plan",
#     "https://www.eda.gov/funding/funding-opportunities/economic-adjustment-assistance-fy21-american-rescue-plan",
#     "https://www.eda.gov/funding/funding-opportunities/fiscal-year-2017-fdi-and-trade-international-engagement-ready",
#     "https://www.eda.gov/funding/funding-opportunities/fiscal-year-2018-university-center-economic-development-program-uc",
#     "https://www.eda.gov/funding/funding-opportunities/fiscal-year-2021-2023-research-and-national-technical-assistance-rnta",
#     "https://www.eda.gov/funding/funding-opportunities/fiscal-year-2021-university-center-economic-development-program-uc",
#     "https://www.eda.gov/funding/funding-opportunities/fiscal-year-2022-university-center-economic-development-program-uc",
#     "https://www.eda.gov/funding/funding-opportunities/fiscal-year-2023-disaster-supplemental",
#     "https://www.eda.gov/funding/funding-opportunities/fiscal-year-2023-stem-talent-challenge",
#     "https://www.eda.gov/funding/funding-opportunities/fiscal-year-2023-university-center-economic-development-uc",
#     "https://www.eda.gov/funding/funding-opportunities/fiscal-year-2024-good-jobs-challenge-nofo",
#     "https://www.eda.gov/funding/funding-opportunities/indigenous-communities-fy21-american-rescue-plan",
#     "https://www.eda.gov/funding/funding-opportunities/recompete-pilot-program-phase-1",
#     "https://www.eda.gov/funding/funding-opportunities/statewide-planning-research-networks-fy21-american-rescue-plan",
#     "https://www.eda.gov/funding/funding-opportunities/tech-hubs-program-phase-1",
#     "https://www.eda.gov/funding/funding-opportunities/tech-hubs-program-phase-2",
#     "https://www.eda.gov/funding/funding-opportunities/travel-tourism-outdoor-recreation-fy21-american-rescue-plan",
#     "https://www.eda.gov/funding/programs/build-to-scale/past-resources",
#     "https://www.eda.gov/funding/programs/stem-challenge",
#     "https://www.eda.gov/sites/default/files/2024-09/FY24_B2S_NOFO_FINAL.pdf",
#     "https://www.eda.gov/sites/default/files/files/oie/ris/2018-RIS-Program-NOFO.pdf",
#     "https://www.hrsa.gov/grants/find-funding/HRSA-24-121",
#     "https://www.hrsa.gov/grants/find-funding/HRSA-26-015",
#     "https://www.idsociety.org/practice-resources/grants-and-funding/microbial-pathogenesis-in-alzheimers-disease-research-grant-program",
#     "https://www.irs.gov/charities-non-profits/private-foundations/grants-to-individuals",
#     "https://www.johnahartford.org/grants-strategy/advocacy-core-support-advancing-health-long-term-care-for-older-adults-care-fund",
#     "https://www.johnahartford.org/grants-strategy/age-friendly_health-systems-award",
#     "https://www.nia.nih.gov/research/applicant-resource-notice-special-interest-nosi-telehealth-people-and-families-living",
#     "https://www.nia.nih.gov/research/grants-funding/announcements",
#     "https://www.point32healthfoundation.org/funding-grants/how-we-fund/",
#     "https://www.racf.org/grant/wayne-county-community-endowment/",
#     "https://www.racf.org/grant/yates-community-endowment/",
#     "https://www.racf.org/grants/grant-opportunities/",
#     "https://www.rrf.org/what-we-fund/social-and-intergenerational-connectedness/",
#     "https://www.tembo.health/post/helping-facilities-apply-for-200m-in-telehealth-funding",
#     "https://www.t-mobile.com/brand/hometown-grants",
#     "https://www.usbank.com/about-us-bank/community/community-possible-grant-program.html",
#     "https://www.who.int/news-room/articles-detail/call-for-applications---2024-hands-on-training-for-mrna-vaccine-manufacturing--organized-by-gth-b-and-supported-by-who",
#     "https://www.who.int/news-room/articles-detail/call-for-applications---2024-hands-on-training-for-upstream-process-in-cell-based-vaccine-manufacturing--organized-by-gth-b-and-supported-by-who",
#     "https://www.who.int/news-room/articles-detail/call-for-applications-2025-hands-on-training-for-antibody-manufacturing-organised-by-the-global-training-hub-for-biomanufacturing-in-the-republic-of-korea--supported-by-the-world-health-organization",
#     "https://www.who.int/news-room/articles-detail/call-for-applications---2025-hands-on-training-for-good-manufacturing-practice-in-biomanufacturing-organised-by-the-global-training-hub-for-biomanufacturing-in-the-republic-of-korea--supported-by-the-world-health-organization",
#     "https://www.who.int/news-room/articles-detail/call-for-applications-2025-hands-on-training-for-mrna-vaccine-manufacturing-organised-by-the-global-training-hub-for-biomanufacturing-in-the-republic-of-korea--supported-by-the-world-health-organization",
#     "https://www.who.int/news-room/articles-detail/call-for-applications---2025-hands-on_training-for-upstream-processin-cell-based-vaccine-manufacturing-organized-by-the-global-training-hub-for-biomanufacturing-(gth-b)in-the-republic-of-korea--supported-by-the-world-health-organization",
#     "https://www.who.int/news-room/articles-detail/call-for-applications---2025-introductory-course-for-biologics-development-and-manufacturing-organised-by-the-global-training-hub-for-biomanufacturing-(gth-b)-in-seoul--republic-of-korea",
#     "https://www.who.int/news-room/articles-detail/call-for-applications---2025-introductory-course-for-standard-practice-organized-by-the-global-training-hub-for-biomanufacturing--in-the-republic-of-korea--supported-by-the-world-health-organization"
# ]

curated_urls = [
    "https://www.johnahartford.org/grants-strategy/age-friendly-health-systems-award"
]



# ----------------------------------------------------
# 5) Main ingest logic: use existing corpus when possible, append to XLSX
# ----------------------------------------------------
if not curated_urls:
    raise ValueError("curated_urls list is empty. Paste your URLs into the list before running.")

if not os.path.exists(EXCEL_PATH):
    raise FileNotFoundError(f"Excel file not found: {EXCEL_PATH}")

# load workbook + all sheets into memory
xls = pd.ExcelFile(EXCEL_PATH)
sheet_names = xls.sheet_names
if not sheet_names:
    raise ValueError(f"No sheets found in {EXCEL_PATH}")

first_sheet_name = sheet_names[0]
df_first = xls.parse(first_sheet_name)

# read ALL other sheets up front, BEFORE we open ExcelWriter in mode="w"
other_sheets = {}
for sname in sheet_names[1:]:
    other_sheets[sname] = xls.parse(sname)

# determine starting scrape_num
if "scrape_num" in df_first.columns:
    max_scrape = pd.to_numeric(df_first["scrape_num"], errors="coerce").max()
    if pd.isna(max_scrape):
        max_scrape = 0
else:
    max_scrape = 0
scrape_counter = int(max_scrape)

# track existing URLs/doc_ids to avoid accidental duplicates in Excel
existing_urls = set(df_first["url"].astype(str)) if "url" in df_first.columns else set()
existing_doc_ids = set(df_first["doc_id"].astype(str)) if "doc_id" in df_first.columns else set()

new_rows = []

for raw_u in curated_urls:
    if not isinstance(raw_u, str) or not raw_u.strip():
        continue

    url = raw_u.strip()
    norm_u = normalize_url(url)

    if norm_u in existing_urls:
        print(f"Skipping already-logged URL (in Excel): {url}")
        continue

    did = doc_id_from_url(url)
    if did in existing_doc_ids:
        print(f"Skipping URL because doc_id already in Excel: {url} ({did})")
        continue

    text_path = os.path.join(CORPUS_TEXT_DIR, f"{did}.txt")
    html_path = os.path.join(CORPUS_HTML_DIR, f"{did}.html")

    print(f"\nCURATED INGEST (Excel-only step, reusing corpus if present): {url}")

    # 1) get clean_text: prefer existing corpus text, else fetch+save
    if os.path.exists(text_path):
        # reuse existing corpus text, NO new corpus addition
        try:
            with open(text_path, "r", encoding="utf-8", errors="ignore") as f:
                clean_text = f.read()
        except Exception as e:
            print(f"  ERROR reading existing text file for {url}: {e}")
            continue
    else:
        # need to fetch and create corpus files
        try:
            r = requests.get(url, headers=HEADERS, timeout=20)
            r.raise_for_status()
        except Exception as e:
            print(f"  ERROR fetching {url}: {e}")
            continue

        soup = BeautifulSoup(r.text, "html.parser")
        html_raw = str(soup)
        clean_text = html_to_clean_text(html_raw)

        if not clean_text:
            print(f"  WARNING: empty text for {url}, skipping.")
            continue

        # save html/txt into corpus (first and only time for this doc_id)
        try:
            with open(html_path, "w", encoding="utf-8", errors="ignore") as f:
                f.write(html_raw)
            with open(text_path, "w", encoding="utf-8", errors="ignore") as f:
                f.write(clean_text)
        except Exception as e:
            print(f"  ERROR writing corpus files for {url}: {e}")
            continue

    # 2) compute scores
    try:
        k_score, matched = keyword_score(clean_text, keyword_weights)
    except Exception as e:
        print(f"  error computing keyword_score for {url}: {e}")
        k_score, matched = 0, []

    try:
        c_score = cosine_score(clean_text)
    except Exception as e:
        print(f"  error computing cosine_score for {url}: {e}")
        c_score = 0.0

    scrape_counter += 1

    matched_set = set(matched)
    aging = 1 if "aging" in matched_set else 0
    dementia = 1 if "dementia" in matched_set else 0
    alzhiemers = 1 if "alzheimer's" in matched_set else 0
    isolation = 1 if "isolation" in matched_set else 0
    telehealth = 1 if "telehealth" in matched_set else 0
    np_501c3 = 1 if "501(c)(3)" in matched_set else 0
    grant_kw = 1 if "grant" in matched_set else 0
    funding_kw = 1 if "funding" in matched_set else 0

    row = {
        "url": norm_u,
        "scrape_num": scrape_counter,
        "content_snippet": clean_text[:2000],
        "score": float(k_score + (c_score or 0.0)),
        "matched_keywords": ", ".join(matched),
        "num_keywords": len(matched),
        "aging": aging,
        "dementia": dementia,
        "alzhiemers": alzhiemers,
        "isolation": isolation,
        "telehealth": telehealth,
        "np_501c3": np_501c3,
        "grant": grant_kw,
        "funding": funding_kw,
        "relevence_label": "",
        "grant_yesno": "grant",  # these are pre-vetted
        "label_note": "",
        "keyword_score": k_score,
        "cosine_similarity": round(float(c_score or 0.0), 6),
        "granting_organization": "",
        "date_of_loi_due": "",
        "date_of_submission": "",
        "doc_id": did,
        "source_domain": urlparse(url).netloc,
        "html_path": os.path.abspath(html_path),
        "content_sha1": content_sha1(clean_text),
        "is_detail_page": 1,
        "text_path": os.path.abspath(text_path),
        "date_seen_utc": now_iso_utc()
    }

    new_rows.append(row)
    existing_urls.add(norm_u)
    existing_doc_ids.add(did)

if not new_rows:
    print("\nNo new rows created from curated URLs (nothing new to add to Excel).")
else:
    df_new = pd.DataFrame(new_rows)

    # align columns between existing first sheet and new rows
    for col in df_first.columns:
        if col not in df_new.columns:
            df_new[col] = ""

    for col in df_new.columns:
        if col not in df_first.columns:
            df_first[col] = ""

    ordered_cols = list(df_first.columns) + [c for c in df_new.columns if c not in df_first.columns]
    df_first = df_first[ordered_cols]
    df_new = df_new[ordered_cols]

    df_first_updated = pd.concat([df_first, df_new], ignore_index=True)

    # write back: first sheet updated, all others preserved (from other_sheets dict)
    with pd.ExcelWriter(EXCEL_PATH, engine="openpyxl", mode="w") as writer:
        # updated first sheet
        df_first_updated.to_excel(writer, sheet_name=first_sheet_name, index=False)
        # all other sheets exactly as they were
        for sname, df_other in other_sheets.items():
            df_other.to_excel(writer, sheet_name=sname, index=False)

    print(f"\nOK: appended {len(df_new)} curated URLs to first sheet '{first_sheet_name}' in {EXCEL_PATH}")
    print("No duplicate corpus additions were made; existing text/html files were reused when present.")


Skipping already-logged URL (in Excel): https://www.johnahartford.org/grants-strategy/age-friendly-health-systems-award

No new rows created from curated URLs (nothing new to add to Excel).


In [ ]:
import os

EXCEL_PATH = r"C:\Users\miked\Desktop2\IConnectFoundation\grantMinded\logs\scored_results.xlsx"

try:
    with open(EXCEL_PATH, "a") as f:
        f.write("")
    print("Python CAN write the file — unlocked.")
except Exception as e:
    print("Python CANNOT write the file —", e)


In [ ]:
# # grantminded_corpus.py
# # corpus builder for grantMinded: scrapes, triages, and saves grant detail pages

# import os
# import re
# import csv
# import json
# import time
# import random
# import hashlib
# import datetime as _dt
# from urllib.parse import urlparse, urlunparse, urljoin

# import requests
# from bs4 import BeautifulSoup
# import pandas as pd

# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.metrics.pairwise import cosine_similarity
# import joblib

# # -----------------------
# # config
# # -----------------------
# HEADERS = {
#     "User-Agent": (
#         "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
#         "AppleWebKit/537.36 (KHTML, like Gecko) "
#         "Chrome/114.0.0.0 Safari/537.36"
#     )
# }

# LOG_PATH = r"C:\Users\miked\Desktop2\IConnectFoundation\grantMinded\logs"
# os.makedirs(LOG_PATH, exist_ok=True)

# CORPUS_HTML_DIR = os.path.join(LOG_PATH, "corpus", "html")
# CORPUS_TEXT_DIR = os.path.join(LOG_PATH, "corpus", "text")
# os.makedirs(CORPUS_HTML_DIR, exist_ok=True)
# os.makedirs(CORPUS_TEXT_DIR, exist_ok=True)

# CSV_PATH = os.path.join(LOG_PATH, "scored_results.csv")

# # path to saved logistic regression grant detector bundle
# GRANT_MODEL_BUNDLE_PATH = r"C:\Users\miked\Desktop2\IConnectFoundation\grantMinded\grant_logreg_bundle.pkl"

# # corpus cap
# MAX_CORPUS_ROWS = 500          # total rows to keep in scored_results.csv
# PREFER_LABELED = True          # keep labeled rows first when capping
# PRUNE_ORPHAN_FILES = False     # leave False unless you want file pruning

# # domain-expert keyword weights (for mission relevance)
# keyword_weights = {
#     "aging": 3,
#     "dementia": 3,
#     "isolation": 4,
#     "alzheimer's": 3,
#     "telehealth": 2,
#     "501(c)(3)": 3,
#     "grant": 4,
#     "funding": 4
# }

# # example web search queries (optional)
# search_keywords = [
#     "aging nonprofit funding",
#     "alzheimer's foundation grant",
#     "dementia telehealth grant",
#     "community isolation grant 501(c)(3)",
#     "social isolation grant"
# ]

# _reference_text = (
#     "health care programs and partnerships that improve outcomes and quality of life for older adults, "
#     "people with disabilities, and caregivers. initiatives to align health and social care, community grants, 501 c 3. "
#     "leveraging technology as a tool for connection, promoting meaningful engagement"
# )

# # crawl config
# deny_list = [
#     "/blog/", "/press/", "/press-release/", "/news/", "/stories/", "/story/",
#     "/about/", "/team/", "/board/", "/contact", "/contact-us",
#     "/privacy", "/terms", "/careers", "/jobs", "/event", "/events",
#     "/media", "/newsletter", "/subscribe", "/search",
#     "/login", "/sign-in", "/signin", "/sign_in",
#     "/cart", "/donate",
# ]

# allow_hint_keywords = [
#     "grant","grants","fund","funds","funding",
#     "opportunity","opportunities","rfp","rfa","rfi",
#     "request-for-proposals","request for proposals",
#     "apply","application","guidelines","program","loi","initiative"
# ]

# BINARY_EXTS = (
#     ".pdf",".doc",".docx",".xls",".xlsx",".ppt",".pptx",
#     ".zip",".rar",".7z",".png",".jpg",".jpeg",".gif",".svg",
#     ".mp3",".mp4",".mov",".avi",".wav"
# )

# # global scrape counter (for scrape_num)
# SCRAPE_COUNTER = 0

# # -----------------------
# # helpers: ids, time, text clean
# # -----------------------
# def normalize_url(u):
#     p = urlparse(u)
#     p = p._replace(fragment="")
#     return urlunparse((p.scheme.lower(), p.netloc.lower(), p.path, p.params, p.query, ""))

# def doc_id_from_url(u):
#     return hashlib.sha1(normalize_url(u).encode("utf-8")).hexdigest()[:16]

# def now_iso_utc():
#     t = _dt.datetime.now(_dt.timezone.utc).replace(microsecond=0)
#     return t.isoformat().replace("+00:00", "Z")

# def html_to_clean_text(html):
#     soup = BeautifulSoup(html, "html.parser")
#     for tag in soup(["script", "style", "noscript"]):
#         tag.decompose()
#     text = soup.get_text(separator=" ", strip=True)
#     return re.sub(r"\s+", " ", text).strip()

# def content_sha1(text):
#     return hashlib.sha1((text or "").encode("utf-8")).hexdigest()

# # -----------------------
# # stage 1: keyword score
# # -----------------------
# def keyword_score(text, keywords):
#     score = 0
#     matched = []
#     t = (text or "").lower()
#     for word, weight in keywords.items():
#         pat = rf"\b{re.escape(word.lower())}\b"
#         if re.search(pat, t):
#             score += weight
#             matched.append(word)
#     return score, matched

# # -----------------------
# # stage 2: cosine score (tf-idf)
# # -----------------------
# def _simple_tokenizer(s):
#     s = re.sub(r"[^a-z0-9\s]", " ", (s or "").lower())
#     toks = s.split()
#     return [t for t in toks if len(t) > 2]

# def cosine_score(grant_text, ref_text=_reference_text):
#     docs = [ref_text, grant_text or ""]
#     vec = TfidfVectorizer(stop_words="english", tokenizer=_simple_tokenizer, token_pattern=None)
#     m = vec.fit_transform(docs)
#     sim = cosine_similarity(m[0:1], m[1:2])[0][0]
#     return float(sim)

# # -----------------------
# # load logistic regression grant detector
# # -----------------------
# GRANT_VECTORIZER = None
# GRANT_SCALER = None
# GRANT_MODEL = None
# GRANT_THRESHOLD = 0.1  # explicit threshold

# try:
#     if os.path.exists(GRANT_MODEL_BUNDLE_PATH):
#         _bundle = joblib.load(GRANT_MODEL_BUNDLE_PATH)
#         GRANT_VECTORIZER = _bundle.get("vectorizer", None)
#         GRANT_SCALER = _bundle.get("scaler", None)
#         GRANT_MODEL = _bundle.get("model", None)
#         if "threshold" in _bundle:
#             # you said to use 0.1; we respect that explicitly
#             GRANT_THRESHOLD = 0.1
#         print(f"Loaded grant LR model bundle from {GRANT_MODEL_BUNDLE_PATH}")
#     else:
#         print(f"WARNING: grant model bundle not found at {GRANT_MODEL_BUNDLE_PATH}")
# except Exception as e:
#     print(f"WARNING: could not load grant model bundle: {e}")

# def is_grant_by_lr(text, threshold=GRANT_THRESHOLD):
#     """
#     Use pretrained logistic regression (TF-IDF + scaler) to decide if text describes a grant.
#     Returns (is_grant_bool, prob).
#     If model missing, falls back to (True, 1.0) so we don't silently drop pages.
#     """
#     if not text or len(text.strip()) < 50:
#         return False, 0.0

#     if GRANT_MODEL is None or GRANT_VECTORIZER is None:
#         # fail-safe: keep the page but warn
#         # print("WARNING: GRANT_MODEL not loaded; treating all pages as grants.")
#         return True, 1.0

#     X = GRANT_VECTORIZER.transform([text])
#     if GRANT_SCALER is not None:
#         X = GRANT_SCALER.transform(X)
#     prob = float(GRANT_MODEL.predict_proba(X)[0][1])
#     return prob >= threshold, prob

# # -----------------------
# # http fetch
# # -----------------------
# def scrape_website(url, timeout=15):
#     try:
#         r = requests.get(url, headers=HEADERS, timeout=timeout, allow_redirects=True)
#         r.raise_for_status()
#         soup = BeautifulSoup(r.text, "html.parser")
#         text = soup.get_text(separator=" ", strip=True)
#         return url, text, soup
#     except Exception as e:
#         print(f"error scraping {url}: {e}")
#         return url, "", None

# def _seed_looks_promising(url: str, title: str = "", text: str = "") -> bool:
#     ul = (url or "").lower()
#     tl = (title or text or "").lower()
#     return any(tok in ul for tok in allow_hint_keywords) or any(tok in tl for tok in allow_hint_keywords)

# # -----------------------
# # web search (ddgs primary -> duckduckgo_search -> googlesearch)
# # -----------------------
# def _norm(u):
#     try:
#         p = urlparse(u)
#         return f"{p.scheme.lower()}://{p.netloc.lower()}{p.path}".rstrip("/")
#     except Exception:
#         return u

# def _limit_per_domain(urls, max_per_domain=5):
#     out, seen = [], {}
#     for u in urls:
#         host = urlparse(u).netloc.lower()
#         seen[host] = seen.get(host, 0) + 1
#         if seen[host] <= max_per_domain:
#             out.append(u)
#     return out

# def search_web(query, max_results=30):
#     urls_raw = []

#     q = query.replace("501(c)(3)", '("501(c)(3)" OR 501c3)')

#     try:
#         from ddgs import DDGS
#         with DDGS() as ddgs:
#             res = list(ddgs.text(q, max_results=max_results*2))
#             urls_raw.extend([(r.get("href"), r.get("title", "")) for r in res if r.get("href")])
#         print(f"ddgs returned {len(urls_raw)} raw results for: {query}")
#     except Exception as e:
#         print(f"ddgs error for '{query}': {e}")
#         try:
#             from duckduckgo_search import DDGS as OldDDGS
#             with OldDDGS() as ddgs:
#                 res = list(ddgs.text(q, max_results=max_results*2))
#                 urls_raw.extend([(r.get("href"), r.get("title", "")) for r in res if r.get("href")])
#             print(f"duckduckgo_search returned {len(urls_raw)} raw results for: {query}")
#         except Exception as e2:
#             print(f"duckduckgo_search error for '{query}': {e2}")

#     if len(urls_raw) < max_results:
#         try:
#             from googlesearch import search as gsearch
#             g = [(u, "") for u in gsearch(q, num_results=max_results) if u]
#             urls_raw.extend(g)
#             print(f"googlesearch returned {len(g)} raw results for: {query}")
#         except Exception as e:
#             print(f"googlesearch blocked/unavailable for '{query}': {e}")

#     filtered = []
#     for href, title in urls_raw:
#         if not href or not href.startswith("http"):
#             continue
#         if _seed_looks_promising(href, title):
#             filtered.append(_norm(href))

#     filtered = list(dict.fromkeys(filtered))
#     filtered = _limit_per_domain(filtered, max_per_domain=5)
#     random.shuffle(filtered)
#     filtered = filtered[:max_results]

#     print(f"using {len(filtered)} urls after filters for: {query}")
#     return filtered

# def google_search(query, num_results=30):
#     return search_web(query, max_results=num_results)

# # -----------------------
# # url helpers
# # -----------------------
# def _strip_www(host: str) -> str:
#     return host[4:] if host.lower().startswith("www.") else host.lower()

# def same_domain(u, base):
#     try:
#         hu = _strip_www(urlparse(u).netloc)
#         hb = _strip_www(urlparse(base).netloc)
#         return hu == hb or hu.endswith("." + hb) or hb.endswith("." + hu)
#     except Exception:
#         return False

# def is_denied(u):
#     p = urlparse(u)
#     path = (p.path or "").lower()

#     for ext in BINARY_EXTS:
#         if path.endswith(ext):
#             return True

#     segs = [s for s in path.split("/") if s]
#     for d in deny_list:
#         d_clean = d.strip("/").lower()
#         if not d_clean:
#             continue
#         if (
#             d_clean in segs or
#             path == f"/{d_clean}" or
#             path.startswith(f"/{d_clean}/")
#         ):
#             return True

#     return False

# def looks_relevant_link(href, text_lower):
#     href_l = (href or "").lower()
#     if any(k in href_l for k in allow_hint_keywords):
#         return True
#     if any(k in (text_lower or "") for k in allow_hint_keywords):
#         return True
#     return False

# # -----------------------
# # link extraction
# # -----------------------
# def extract_links(current_url, soup, base_url, generic_cap=20):
#     links_priority, links_generic = [], []
#     if soup is None:
#         return []

#     for a in soup.find_all("a", href=True):
#         href = a.get("href") or ""
#         href_l = href.lower()

#         if href_l.startswith("#") or href_l.startswith("mailto:") \
#            or href_l.startswith("tel:") or href_l.startswith("javascript:"):
#             continue

#         link = urljoin(current_url, href)
#         if not link.startswith(("http://","https://")):
#             continue
#         if not same_domain(link, base_url):
#             continue
#         if is_denied(link):
#             continue

#         txt = (a.get_text(strip=True) or "").lower()

#         is_pagination = bool(
#             re.search(r"(next|older|previous|more|>>|«|»|‹|›|load more|page\s*\d+)", txt)
#             or re.search(r"(?:[?&](?:page|p|start|offset)=\d+)", link, re.I)
#         )

#         if looks_relevant_link(href, txt) or is_pagination:
#             links_priority.append(link)
#         else:
#             links_generic.append(link)

#     seen, out = set(), []
#     for u in links_priority + links_generic[:generic_cap]:
#         if u not in seen:
#             seen.add(u)
#             out.append(u)
#     return out

# # -----------------------
# # save detail page + build row
# # -----------------------
# def save_detail_page_and_row(url, soup, k_score, matched_list, c_score, scrape_num):
#     html_raw = str(soup)
#     clean_text = html_to_clean_text(html_raw)

#     did = doc_id_from_url(url)
#     html_path = os.path.join(CORPUS_HTML_DIR, f"{did}.html")
#     text_path = os.path.join(CORPUS_TEXT_DIR, f"{did}.txt")

#     with open(html_path, "w", encoding="utf-8", errors="ignore") as f:
#         f.write(html_raw)
#     with open(text_path, "w", encoding="utf-8", errors="ignore") as f:
#         f.write(clean_text)

#     # per-keyword binary flags
#     matched_set = set(matched_list)
#     aging = 1 if "aging" in matched_set else 0
#     dementia = 1 if "dementia" in matched_set else 0
#     alzhiemers = 1 if "alzheimer's" in matched_set else 0
#     isolation = 1 if "isolation" in matched_set else 0
#     telehealth = 1 if "telehealth" in matched_set else 0
#     np_501c3 = 1 if "501(c)(3)" in matched_set else 0
#     grant_kw = 1 if "grant" in matched_set else 0
#     funding_kw = 1 if "funding" in matched_set else 0

#     row = {
#         "url": normalize_url(url),
#         "scrape_num": scrape_num,
#         "content_snippet": clean_text[:2000],
#         "score": float(k_score + (c_score or 0.0)),
#         "matched_keywords": ", ".join(matched_list),
#         "num_keywords": len(matched_list),
#         "aging": aging,
#         "dementia": dementia,
#         "alzhiemers": alzhiemers,
#         "isolation": isolation,
#         "telehealth": telehealth,
#         "np_501c3": np_501c3,
#         "grant": grant_kw,
#         "funding": funding_kw,
#         "relevence_label": "",        # to be hand-labeled later
#         "grant_yesno": "grant",       # only saving pages classified as grants
#         "label_note": "",
#         "keyword_score": k_score,
#         "cosine_similarity": round(float(c_score or 0.0), 6),
#         "granting_organization": "",
#         "date_of_loi_due": "",
#         "date_of_submission": "",
#         "doc_id": did,
#         "source_domain": urlparse(url).netloc,
#         "html_path": os.path.abspath(html_path),
#         "content_sha1": content_sha1(clean_text),
#         "is_detail_page": 1,
#         "text_path": os.path.abspath(text_path),

#         # extra meta (not in your required list but useful for sorting)
#         "date_seen_utc": now_iso_utc()
#     }
#     return row

# # -----------------------
# # csv helpers
# # -----------------------
# def read_csv_safe(path):
#     for enc in ("utf-8", "utf-8-sig", "latin1", "cp1252"):
#         try:
#             return pd.read_csv(path, encoding=enc)
#         except UnicodeDecodeError:
#             continue
#     return pd.read_csv(path, encoding="utf-8", encoding_errors="replace")

# def existing_keys(df, key_cols):
#     if not set(key_cols).issubset(df.columns):
#         return set()
#     if len(key_cols) == 1:
#         return set(df[key_cols[0]].astype(str))
#     return set(map(tuple, df[key_cols].astype(str).itertuples(index=False, name=None)))

# def enforce_corpus_cap(df_all: pd.DataFrame,
#                        max_rows: int,
#                        prefer_labeled: bool = True) -> pd.DataFrame:
#     if max_rows is None or max_rows <= 0 or len(df_all) <= max_rows:
#         return df_all

#     rel = df_all["relevance"].astype(str).str.strip().fillna("") if "relevance" in df_all.columns else pd.Series([""]*len(df_all))
#     vlabel = df_all["valid_label"].astype(str).str.strip().fillna("") if "valid_label" in df_all.columns else pd.Series([""]*len(df_all))
#     is_labeled = (rel != "") | (vlabel.str.lower() != "unlabeled")

#     if prefer_labeled and is_labeled.any():
#         df_lab = df_all[is_labeled].copy()
#         df_unl = df_all[~is_labeled].copy()

#         if len(df_lab) >= max_rows:
#             return df_lab.tail(max_rows).copy()

#         remaining = max_rows - len(df_lab)
#         return pd.concat([df_lab, df_unl.tail(remaining)], ignore_index=True)
#     else:
#         return df_all.tail(max_rows).copy()

# def prune_orphan_files(df_all: pd.DataFrame):
#     if "doc_id" not in df_all.columns:
#         print("prune_orphan_files: skipped (no 'doc_id' column).")
#         return

#     keep = set(df_all["doc_id"].astype(str))
#     removed = 0

#     def _rm(path):
#         nonlocal removed
#         try:
#             if os.path.exists(path):
#                 os.remove(path)
#                 removed += 1
#         except Exception as e:
#             print(f"  could not remove {path}: {e}")

#     for fname in os.listdir(CORPUS_HTML_DIR):
#         if not fname.lower().endswith(".html"):
#             continue
#         did = os.path.splitext(fname)[0]
#         if did not in keep:
#             _rm(os.path.join(CORPUS_HTML_DIR, fname))

#     for fname in os.listdir(CORPUS_TEXT_DIR):
#         if not fname.lower().endswith(".txt"):
#             continue
#         did = os.path.splitext(fname)[0]
#         if did not in keep:
#             _rm(os.path.join(CORPUS_TEXT_DIR, fname))

#     if removed:
#         print(f"prune_orphan_files: removed {removed} old files.")

# # -----------------------
# # crawler
# # -----------------------
# def crawl_site(start_url, max_pages=300, max_depth=3):
#     visited = set()
#     results = []

#     def crawl(url, depth_left):
#         nonlocal results
#         global SCRAPE_COUNTER

#         if depth_left < 0 or len(visited) >= max_pages:
#             return
#         if url in visited:
#             return
#         visited.add(url)

#         print(f"visiting {url} (depth {depth_left})")
#         url, raw_text, soup = scrape_website(url)
#         if not soup or not raw_text:
#             return

#         # triage: keyword + cosine
#         try:
#             k_score, matched = keyword_score(raw_text, keyword_weights)
#             c_score = cosine_score(raw_text)
#         except Exception as e:
#             print(f"  error computing keyword/cosine for {url}: {e}")
#             k_score, matched, c_score = 0, [], 0.0

#         # only consider for saving if triage passes
#         if (k_score >= 7) or (c_score >= 0.07):
#             clean_text = html_to_clean_text(str(soup))
#             is_grant, prob = is_grant_by_lr(clean_text, threshold=GRANT_THRESHOLD)
#             if is_grant:
#                 SCRAPE_COUNTER += 1
#                 print(
#                     f"  SAVED grant (scrape_num={SCRAPE_COUNTER}): "
#                     f"keyword_score={k_score}, cosine={round(c_score,3)}, prob={round(prob,3)}, matched={matched}"
#                 )
#                 row = save_detail_page_and_row(
#                     url, soup, k_score, matched, c_score, scrape_num=SCRAPE_COUNTER
#                 )
#                 results.append(row)
#             else:
#                 print(
#                     f"  triage passed but LR says NOT GRANT: "
#                     f"keyword_score={k_score}, cosine={round(c_score,3)}, prob={round(prob,3)}"
#                 )
#         else:
#             print(
#                 f"  triage failed: keyword_score={k_score}, cosine={round(c_score,3)}"
#             )

#         # crawl links regardless of whether we saved this page
#         child_links = extract_links(url, soup, start_url)
#         for nxt in child_links[:50]:
#             crawl(nxt, depth_left - 1)

#     crawl(start_url, max_depth)
#     return results

# # -----------------------
# # pipeline
# # -----------------------
# def run_pipeline(depth=3):
#     results = []

#     static_urls = [
#         "https://www.rwjf.org/en/grants/active-funding-opportunities.html",
#         "https://www.eda.gov/funding/funding-opportunities",
#         "https://www.walmart.org/how-we-give/open-applications",
#         "https://www.gatesfoundation.org/about/how-we-work/grant-opportunities"
#     ]

#     # crawl static seeds
#     for url in static_urls:
#         print(f"\nSCRAPING LISTING: {url}")
#         try:
#             grant_rows = crawl_site(url, max_pages=600, max_depth=depth)
#             results.extend(grant_rows)
#         except Exception as e:
#             print(f"error on static url {url}: {e}")

#     # discovery via web search
#     for query in search_keywords:
#         print(f"\nSEARCHING WEB FOR: '{query}'")
#         try:
#             found = search_web(query, max_results=30)
#         except Exception as e:
#             print(f"search error for '{query}': {e}")
#             found = []

#         for url in found:
#             print(f"  seed from search: {url}")
#             try:
#                 grant_rows = crawl_site(url, max_pages=400, max_depth=depth)
#                 results.extend(grant_rows)
#             except Exception as e:
#                 print(f"error on discovered url {url}: {e}")
#             time.sleep(0.5)

#     # build dataframe for new results
#     df_new = pd.DataFrame(results)

#     # enforce column order and fill missing columns
#     desired_cols = [
#         "url",
#         "scrape_num",
#         "content_snippet",
#         "score",
#         "matched_keywords",
#         "num_keywords",
#         "aging",
#         "dementia",
#         "alzhiemers",
#         "isolation",
#         "telehealth",
#         "np_501c3",
#         "grant",
#         "funding",
#         "relevence_label",
#         "grant_yesno",
#         "label_note",
#         "keyword_score",
#         "cosine_similarity",
#         "granting_organization",
#         "date_of_loi_due",
#         "date_of_submission",
#         "doc_id",
#         "source_domain",
#         "html_path",
#         "content_sha1",
#         "is_detail_page",
#         "text_path"
#     ]

#     # if CSV exists, append
#     if os.path.exists(CSV_PATH):
#         df_old = read_csv_safe(CSV_PATH)

#         # align columns both ways
#         for col in desired_cols:
#             if col not in df_old.columns:
#                 df_old[col] = ""
#             if col not in df_new.columns:
#                 df_new[col] = ""

#         # keep any extra columns (like date_seen_utc) if already present
#         for col in df_old.columns:
#             if col not in df_new.columns:
#                 df_new[col] = ""

#         # choose uniqueness key
#         if "doc_id" in df_old.columns and "doc_id" in df_new.columns:
#             key_cols = ["doc_id"]
#         elif "content_sha1" in df_old.columns and "content_sha1" in df_new.columns:
#             key_cols = ["content_sha1"]
#         else:
#             key_cols = ["url"]

#         old = existing_keys(df_old, key_cols)

#         if len(key_cols) == 1:
#             df_new_unique = df_new[~df_new[key_cols[0]].astype(str).isin(old)].copy()
#         else:
#             df_new_unique = df_new[
#                 ~df_new[key_cols].astype(str).apply(tuple, axis=1).isin(old)
#             ].copy()

#         n_added = len(df_new_unique)
#         df_all = pd.concat([df_old, df_new_unique], ignore_index=True)
#     else:
#         # first run: everything is new
#         # ensure all desired columns present
#         for col in desired_cols:
#             if col not in df_new.columns:
#                 df_new[col] = ""
#         df_all = df_new
#         n_added = len(df_new)

#     # sort by date_seen_utc if present
#     if "date_seen_utc" in df_all.columns:
#         df_all = df_all.sort_values("date_seen_utc")
#     df_all = df_all.reset_index(drop=True)

#     # enforce overall cap
#     max_rows = MAX_CORPUS_ROWS
#     prefer_labeled = PREFER_LABELED
#     prune_files = PRUNE_ORPHAN_FILES

#     if max_rows:
#         before_cap = len(df_all)
#         df_all = enforce_corpus_cap(df_all, max_rows, prefer_labeled=prefer_labeled)
#         after_cap = len(df_all)
#         if after_cap < before_cap:
#             print(
#                 f"Corpus capped at {max_rows} rows "
#                 f"(dropped {before_cap - after_cap} oldest rows"
#                 f"{' while preserving labeled' if prefer_labeled else ''})."
#             )

#         if prune_files:
#             prune_orphan_files(df_all)

#     # enforce final column order for save
#     for col in desired_cols:
#         if col not in df_all.columns:
#             df_all[col] = ""
#     df_all = df_all[desired_cols + [c for c in df_all.columns if c not in desired_cols]]

#     df_all.to_csv(CSV_PATH, index=False, encoding="utf-8")
#     print(f"\nOK: appended {n_added} new rows; total {len(df_all)} rows saved to {CSV_PATH}")

# # -----------------------
# # entry point
# # -----------------------
# if __name__ == "__main__":
#     run_pipeline(depth=4)
